# Estudio de corresistencia entre familias de antibióticos

Este experimento amplía la prueba preliminar conservando su cohorte de E. coli
canina de origen urinario y sus seis variables. La pregunta es qué reglas entre
familias superan controles temporales y de laboratorio y pueden mostrarse en un
explorador educativo. Excluir fármacos de la misma familia reduce redundancia;
**no demuestra novedad clínica**.

Los parámetros y criterios se escribieron en `protocolo_estudio_v1.json` antes
de ejecutar este estudio. Todos los años ya se habían inspeccionado en la prueba
preliminar: **la comprobación de 2023–2024 es exploratoria, no una prueba inédita**.

Se mantienen las fuentes [FDA Vet-LIRN](https://www.fda.gov/animal-veterinary/science-research/veterinary-laboratory-investigation-and-response-network)
y [USDA APHIS 2020](https://usbiotechnologyregulation.mrp.usda.gov/sites/default/files/2020_APHIS_AMR_PilotProject_EOY_Report.pdf).
R usa los cortes históricos del protocolo original. Por debajo de R incluye
intermedios y no equivale a susceptible. No hay predicción de tratamiento.


In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "experiments/protocolo_estudio_v1.json").exists())
sys.path.insert(0, str(root / "experiments"))
import estudio_corresistencia as study

REGENERAR = False  # True vuelve a ejecutar el estudio desde la cohorte verificada.
if REGENERAR:
    study.run()
protocol, original, cohort, matrix, splits = study.load_inputs()
output = root / "results/estudio_v1"
summary = json.loads((output / "resumen.json").read_text(encoding="utf-8"))
model = json.loads((output / "modelo.json").read_text(encoding="utf-8"))
rules = json.loads((output / "reglas_completas.json").read_text(encoding="utf-8"))
assert summary["protocol"] == protocol
print("Hashes de cohorte, protocolo original y archivo oficial verificados.")
print("Muestras:", len(cohort), "| Laboratorios:", cohort["lab"].nunique())
print("Estado de la evaluación:", protocol["evaluation_status"])


Hashes de cohorte, protocolo original y archivo oficial verificados.
Muestras: 3933 | Laboratorios: 25
Estado de la evaluación: Exploratorio: todos estos años ya aparecieron en la prueba preliminar. La confirmación no es una prueba independiente inédita.


## Separación temporal y alternativas

Descubrimiento usa 2017–2021; selección de configuración usa 2022. La publicación
se decide con esos datos y luego se calculan métricas de 2023–2024 sin filtrar
ni ordenar por ellas. Los identificadores de muestra son disjuntos entre etapas.
No podemos descartar que un animal contribuya muestras en varias etapas.

Apriori admite conjuntos de hasta tres ítems. El consecuente pertenece a una
familia ausente del antecedente y el antecedente tiene como máximo un fármaco
por familia. Se exigen al menos 20 coincidencias y 30 antecedentes en descubrimiento.


In [2]:
print(pd.DataFrame([
    {"Etapa": name, "Años": ", ".join(map(str, protocol["periods"][name])), "Muestras": len(part)}
    for name, part in splits.items()
]).to_string(index=False))
print("\nConfiguraciones de búsqueda:")
print(pd.DataFrame(protocol["search"]["configurations"]).to_string(index=False))
print("\nCriterios comunes para evaluar 2022:")
print(json.dumps(protocol["selection"]["common_2022_thresholds"], indent=2))
print("\nControles de laboratorio en descubrimiento:")
print(json.dumps(protocol["selection"]["lab_thresholds_on_discovery"], indent=2))


       Etapa                         Años  Muestras
   discovery 2017, 2018, 2019, 2020, 2021      2193
   selection                         2022       585
confirmation                   2023, 2024      1155

Configuraciones de búsqueda:
        name  min_support  min_confidence
      amplia        0.005             0.4
conservadora        0.010             0.6

Criterios comunes para evaluar 2022:
{
  "min_confidence": 0.5,
  "min_lift": 1.5,
  "min_joint_count": 8,
  "min_antecedent_count": 10
}

Controles de laboratorio en descubrimiento:
{
  "min_labs_with_5_antecedent_samples": 3,
  "min_leave_one_lab_out_lift": 1.2
}


## Comparación y elección de configuración

Primero se compara la cantidad de direcciones entre familias respaldadas por
reglas elegibles. Luego, la proporción de candidatas que cumple los controles.
Se resuelve un empate completo con la configuración conservadora. Estos criterios
valoran cobertura de patrones y estabilidad, en vez de maximizar confianza sola.

Una regla es elegible si en 2022 tiene confianza >=0,50, lift >=1,5, al menos
8 coincidencias y 10 antecedentes; además, al menos 3 laboratorios aportan
5 antecedentes cada uno en descubrimiento y el lift al excluir cada laboratorio
permanece >=1,2. Los umbrales son decisiones exploratorias, no garantías clínicas.


In [3]:
comparison = pd.DataFrame(model["comparison"])
print(comparison.to_string(index=False))
print("\nConfiguración elegida:", summary["winner"])
print("Reglas publicadas:", summary["published_rules"])
rows = [{"Regla": r["id"], "Configuración": r["configuration"],
         "Pasa 2022": r["passes_selection_2022"], "Pasa laboratorios": r["passes_lab_checks"],
         "Elegible": r["eligible"], "Confianza 2022": r["selection"]["confidence"],
         "Coincidencias 2022": r["selection"]["joint_count"]} for r in rules]
print("\nSe conservan también las candidatas descartadas:")
print(pd.DataFrame(rows).to_string(index=False))


configuration  min_support  min_confidence  candidate_rules  passes_2022  eligible_rules  eligible_family_directions  retention_rate
       amplia        0.005             0.4               20            2               2                           1        0.100000
 conservadora        0.010             0.6               14            2               2                           1        0.142857

Configuración elegida: conservadora
Reglas publicadas: 2

Se conservan también las candidatas descartadas:
       Regla Configuración  Pasa 2022  Pasa laboratorios  Elegible  Confianza 2022  Coincidencias 2022
    GEN__FOV        amplia      False               True     False        0.333333                   6
    GEN__CPD        amplia      False               True     False        0.333333                   6
    GEN__EFX        amplia       True               True      True        0.555556                  10
    GEN__MAR        amplia       True               True      True        0.55555

## Resultados seleccionados y comprobación posterior

Las dos reglas publicadas parten de gentamicina y terminan en enrofloxacino o
marbofloxacino. Representan **una misma dirección entre familias**; los consecuentes
pertenecen a la misma familia, de modo que no son dos hallazgos independientes.

La confianza de selección se apoya en solo 18 antecedentes de 2022. Se muestran
los recuentos y los intervalos Wilson para hacer visible esa incertidumbre.
Los intervalos son descriptivos y no corrigen dependencia ni comparaciones múltiples.


In [4]:
for rule in model["rules"]:
    print("\n", rule["id"], "|", rule["family_direction"])
    for stage in ["discovery", "selection", "confirmation"]:
        m = rule[stage]
        print(f"{stage}: {m['joint_count']}/{m['antecedent_count']}; confianza {m['confidence']:.1%}; soporte {m['support']:.1%}; lift {m['lift']:.2f}; intervalo {m['confidence_ci95_low']:.1%}–{m['confidence_ci95_high']:.1%}")
    for year, m in rule["confirmation_by_year"].items():
        print(f"{year}: {m['joint_count']}/{m['antecedent_count']}; confianza {m['confidence']:.1%}")



 GEN__EFX | Aminoglucósidos -> Fluoroquinolonas
discovery: 45/75; confianza 60.0%; soporte 2.1%; lift 6.42; intervalo 48.7%–70.3%
selection: 10/18; confianza 55.6%; soporte 1.7%; lift 5.70; intervalo 33.7%–75.4%
confirmation: 26/37; confianza 70.3%; soporte 2.3%; lift 7.66; intervalo 54.2%–82.5%
2023: 12/19; confianza 63.2%
2024: 14/18; confianza 77.8%

 GEN__MAR | Aminoglucósidos -> Fluoroquinolonas
discovery: 45/75; confianza 60.0%; soporte 2.1%; lift 6.45; intervalo 48.7%–70.3%
selection: 10/18; confianza 55.6%; soporte 1.7%; lift 5.70; intervalo 33.7%–75.4%
confirmation: 27/37; confianza 73.0%; soporte 2.3%; lift 7.95; intervalo 57.0%–84.6%
2023: 13/19; confianza 68.4%
2024: 14/18; confianza 77.8%


## Sensibilidad al laboratorio

Se calcula la regla por laboratorio y se vuelve a calcular en descubrimiento
excluyendo cada laboratorio. Esto comprueba si la asociación agrupada desaparece
al retirar un centro. No asegura estabilidad clínica, ni que todos los laboratorios
observen el mismo patrón. Muchos centros tienen pocos antecedentes.


In [5]:
local = pd.read_csv(output / "reglas_por_laboratorio.csv", dtype={"lab": str})
for rule in model["rules"]:
    print("\n", rule["id"])
    print(json.dumps(rule["lab_audit"], ensure_ascii=False, indent=2))
    table = local[(local["rule_id"] == rule["id"]) & (local["antecedent_count"] >= 5)]
    print(table[["lab", "n", "antecedent_count", "joint_count", "confidence", "lift"]].to_string(index=False))

# Contraste independiente de los recuentos publicados, a partir de los IDs.
for rule in model["rules"]:
    for stage, part in splits.items():
        a_ids = set(part.index)
        for code in rule["antecedent"]:
            a_ids &= set(part.index[part[code]])
        b_ids = set(part.index[part[rule["consequent"]]])
        assert len(a_ids) == rule[stage]["antecedent_count"]
        assert len(a_ids & b_ids) == rule[stage]["joint_count"]
print("\nRecuentos publicados comprobados por intersección de Sample Id.")



 GEN__EFX
{
  "discovery_labs_total": 22,
  "labs_with_at_least_5_antecedents": 6,
  "largest_lab_antecedent_share": 0.10666666666666667,
  "leave_one_lab_out_min_lift": 6.129548122065728,
  "leave_one_lab_out_min_confidence": 0.5774647887323944
}
                                                                                         lab   n  antecedent_count  joint_count  confidence      lift
                               NC State University Veterinary Hospital Diagnostic Laboratory 140                 5            4    0.800000  5.600000
                                               North Dakota Veterinary Diagnostic Laboratory 187                 6            2    0.333333 12.466667
                                               Oklahoma Animal Disease Diagnostic Laboratory 109                 8            6    0.750000  3.406250
  Texas A&M University Veterinary Medical Teaching Hospital Clinical Microbiology Laboratory 119                 7            5    0.714286  5.000000
U

## Integración en la aplicación

La aplicación carga `results/estudio_v1/modelo.json`, permite seleccionar ítems
R y devuelve reglas cuyos antecedentes están en esa selección. Omite consecuentes
que el usuario ya seleccionó. No interpreta ítems no marcados como susceptibles.
Una consulta sin reglas comunica ausencia de reglas seleccionadas, no ausencia
de resistencia ni de asociaciones en general.

Se muestran confianza, soporte, lift, recuentos, intervalos, períodos, procedencia
y limitaciones. Solo se ofrecen reglas de la configuración elegida que cumplen
los controles. No se almacenan registros de pacientes.

Reproducir desde la raíz del proyecto:

```bash
python experiments/primera_prueba.py
python experiments/estudio_corresistencia.py
python experiments/generar_notebook_estudio.py
python -m unittest discover -s tests -v
python app/server.py --port 8787
```

La selección es prometedora para un explorador académico y sigue teniendo respaldo
limitado. El proyecto requiere revisión de Pablo Valenzuela y las especificaciones
de las entregas. Una futura evaluación externa y nuevos datos serían necesarios
para sostener generalización fuera de esta red de vigilancia.
